In [1]:
import pandas as pd
from sqlalchemy import create_engine, URL
from getpass import getpass

url = URL.create(
    "postgresql+psycopg",
    username="postgres",          # from pgAdmin
    password=getpass("Postgres password: "),
    host="localhost",
    port=5432,                    # from pgAdmin
    database="olist",
)
engine = create_engine(url)

In [2]:
df = pd.read_sql("SELECT * FROM analysis_orders", engine)
df.shape

(95824, 13)

In [3]:
# How are review scores distributed for late vs on-time orders?
print(df.groupby("is_late")["review_score"].agg(["count", "mean"]).round(2))

pd.crosstab(df["is_late"], df["review_score"], normalize="index").round(3)

         count  mean
is_late             
False    89443  4.29
True      6381  2.27


review_score,1,2,3,4,5
is_late,,,,,
False,0.066,0.026,0.081,0.204,0.623
True,0.538,0.087,0.109,0.102,0.165


In [4]:
from scipy import stats

late   = df.loc[df["is_late"],  "review_score"]
ontime = df.loc[~df["is_late"], "review_score"]

u, p = stats.mannwhitneyu(late, ontime, alternative="two-sided")

# Effect size: rank-biserial correlation (from -1 to 1; 0 = no difference)
rank_biserial = 1 - 2 * u / (len(late) * len(ontime))

print(f"Mann-Whitney U = {u:,.0f}, p = {p:.3g}")
print(f"Rank-biserial effect size = {rank_biserial:.3f}")

Mann-Whitney U = 103,305,546, p = 0
Rank-biserial effect size = 0.638


In [5]:
import numpy as np

rng = np.random.default_rng(42)          # fixed seed so results are reproducible
late_arr, ontime_arr = late.to_numpy(), ontime.to_numpy()

diffs = []
for _ in range(2000):
    l = rng.choice(late_arr, size=len(late_arr), replace=True)
    o = rng.choice(ontime_arr, size=len(ontime_arr), replace=True)
    diffs.append(o.mean() - l.mean())

low, high = np.percentile(diffs, [2.5, 97.5])
print(f"Gap = {ontime_arr.mean() - late_arr.mean():.2f} stars, 95% CI [{low:.2f}, {high:.2f}]")

Gap = 2.02 stars, 95% CI [1.98, 2.06]


In [6]:
import statsmodels.formula.api as smf

df["is_late_int"] = df["is_late"].astype(int)
df["log_price"]   = np.log(df["order_price"])

# Model 1: lateness only (should match the 2.02-star gap)
m1 = smf.ols("review_score ~ is_late_int", data=df).fit(cov_type="HC1")

# Model 2: lateness plus controls
m2 = smf.ols(
    "review_score ~ is_late_int + log_price + order_freight + n_items + n_sellers"
    " + C(customer_state) + C(category)",
    data=df,
).fit(cov_type="HC1")

for name, m in [("Lateness only", m1), ("With controls", m2)]:
    low, high = m.conf_int().loc["is_late_int"]
    print(f"{name}: late effect = {m.params['is_late_int']:.2f} stars, "
          f"95% CI [{low:.2f}, {high:.2f}]")

Lateness only: late effect = -2.02 stars, 95% CI [-2.06, -1.98]
With controls: late effect = -2.01 stars, 95% CI [-2.05, -1.97]


In [7]:
print(m2.params[["n_items", "n_sellers"]].round(2))
print(df.groupby("n_sellers")[["is_late_int", "review_score"]].mean().round(2).head(3))

n_items     -0.22
n_sellers   -1.01
dtype: float64
           is_late_int  review_score
n_sellers                           
1                 0.07          4.17
2                 0.01          2.89
3                 0.00          2.35


## Findings: Do late deliveries hurt review scores?

**Data:** 95,824 delivered orders with a review (one review per order, most recent kept).
6,381 orders (6.7%) arrived after the delivery date promised at checkout.

**Late orders get far worse reviews.** Late orders average 2.27 stars vs 4.29 for on-time orders,
a gap of 2.02 stars (bootstrapped 95% CI: 1.98–2.06). More than half of late orders (54%) got
1 star, compared with 7% of on-time orders. On-time orders got 5 stars 62% of the time vs 17% for late ones.

**The effect is large, not just statistically significant.** A Mann-Whitney test gives p < 0.001,
but with 95,000 orders almost any difference would be significant. The effect size is what matters:
rank-biserial = 0.64 (large). Put simply, if you pick one late and one on-time order at random,
the late order has the higher score only 18% of the time.

**The gap is not explained by other factors.** A regression controlling for customer state,
product category, price, freight cost, number of items and number of sellers barely changes the
estimate: late orders score 2.01 stars lower (95% CI: 1.97–2.05). Using each order's first review instead of the most recent gives the same result (−2.01 stars).

**Hidden lateness in multi-seller orders.** Each additional seller in an order is associated with
reviews about 1 star lower, even though multi-seller orders are rarely marked late. These orders
usually ship in separate parcels, and the dataset records a single delivery date per order, so a
missing second parcel likely goes unrecorded. Review comments mentioning items still not received
support this.

**Limitations.** This is observational data, so it shows a strong association, not proof that
lateness causes low reviews; unmeasured factors (e.g. a seller who is both slow and sells
poor-quality products) could still contribute. The late measure likely misses late parcels in
multi-seller orders (~1% of orders). Customers who leave reviews may differ from those who don't:
people with strong feelings about the product or delivery are more likely to respond, and less
digitally comfortable customers (e.g. some seniors) may not review at all. Almost every order in
this dataset (99.2%) has a review, which is unusually high and suggests the public data may have
been pre-filtered toward reviewed orders, so results may not fully represent all Olist customers.

In [8]:
# Robustness check: keep each order's FIRST review instead of the most recent
first_reviews = pd.read_sql("""
    SELECT order_id, review_score AS first_score
    FROM (
        SELECT order_id, review_score,
               ROW_NUMBER() OVER (PARTITION BY order_id
                                  ORDER BY review_answer_timestamp ASC, review_id) AS rn
        FROM order_reviews
    ) r
    WHERE rn = 1
""", engine)

df_first = df.merge(first_reviews, on="order_id")

# How many orders actually changed score?
print("Orders where first and latest score differ:",
      (df_first["first_score"] != df_first["review_score"]).sum())

# Raw gap with first reviews
print(df_first.groupby("is_late")["first_score"].mean().round(2))

# Same regression as before, but with first_score
m2_first = smf.ols(
    "first_score ~ is_late_int + log_price + order_freight + n_items + n_sellers"
    " + C(customer_state) + C(category)",
    data=df_first,
).fit(cov_type="HC1")
low, high = m2_first.conf_int().loc["is_late_int"]
print(f"First-review version: late effect = {m2_first.params['is_late_int']:.2f} stars, "
      f"95% CI [{low:.2f}, {high:.2f}]")

Orders where first and latest score differ: 188
is_late
False    4.29
True     2.27
Name: first_score, dtype: float64
First-review version: late effect = -2.01 stars, 95% CI [-2.05, -1.97]


In [9]:
# Map each state to its region
regions = {
    "North":        ["AC", "AM", "AP", "PA", "RO", "RR", "TO"],
    "Northeast":    ["AL", "BA", "CE", "MA", "PB", "PE", "PI", "RN", "SE"],
    "Central-West": ["DF", "GO", "MT", "MS"],
    "Southeast":    ["ES", "MG", "RJ", "SP"],
    "South":        ["PR", "RS", "SC"],
}
state_to_region = {s: r for r, states in regions.items() for s in states}
df["region"] = df["customer_state"].map(state_to_region)

# Look first: median freight cost and freight as a share of price, by region
df["freight_share"] = df["order_freight"] / df["order_price"]
print(df.groupby("region")[["order_freight", "freight_share"]].median().round(2)
        .sort_values("order_freight"))

# Test: do the regions differ?
groups = [g["order_freight"].to_numpy() for _, g in df.groupby("region")]
h, p = stats.kruskal(*groups)

# Effect size: epsilon-squared (0 = no difference, 1 = regions fully separate)
n, k = len(df), len(groups)
eps_sq = (h - k + 1) / (n - k)
print(f"Kruskal-Wallis H = {h:,.1f}, p = {p:.3g}, epsilon-squared = {eps_sq:.3f}")

              order_freight  freight_share
region                                    
Southeast             15.56           0.21
South                 18.23           0.25
Central-West          18.85           0.25
Northeast             26.99           0.32
North                 31.52           0.36
Kruskal-Wallis H = 15,242.8, p = 0, epsilon-squared = 0.159


### Shipping cost by region

**Prediction (before running):** The North will pay the most for shipping because it's farthest
from the São Paulo sellers and has poor road access (much of it is Amazon rainforest).
The Southeast will pay the least because most sellers are located there, so parcels travel
the shortest distance. The South should be close behind thanks to good infrastructure.

My prediction was half right: the North pays the most for shipping, but the Southeast (not the South) pays the least, likely because most sellers are based there. Median shipping in the North is R$31.52 vs R$15.56 in the Southeast — about double — and takes 36% of the item price vs 21%. The regional difference is substantial (epsilon-squared = 0.16). Combined with the Northeast’s high late rates, customers in the north and northeast pay the most to wait the longest.